In [ ]:
import sys
print(sys.executable)

In [ ]:
import requests
import pandas as pd

APP_ID = ""   # 填入你的 Adzuna app_id
APP_KEY = ""  # 填入你的 Adzuna app_key

In [ ]:
url = "https://api.adzuna.com/v1/api/jobs/gb/search/1"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data analyst",
    "where": "london",
}

r = requests.get(url, params=params)
print(r.status_code)

data = r.json()
print(data["count"])

In [ ]:
import sys
print(sys.executable)

In [ ]:
url = "https://api.adzuna.com/v1/api/jobs/gb/search/2"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data analyst",
    "where": "london",
}

r = requests.get(url, params=params)
print(r.status_code)

data = r.json()
print(data["count"])

In [ ]:
data = r.json()
print(data["results"][0]["title"])
print(data["results"][0]["id"])

In [ ]:
import time

all_jobs = []

for page in range(1, 6):
    url = f"https://api.adzuna.com/v1/api/jobs/gb/search/{page}"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 50,
        "what": "data analyst",
        "where": "london",
    }
    r = requests.get(url, params=params)
    print(f"page {page}: {r.status_code}")
    
    data = r.json()
    all_jobs.extend(data["results"])
    
    time.sleep(1)

print(f"总共拿到 {len(all_jobs)} 条")

In [ ]:
df = pd.DataFrame(all_jobs)
print(df.shape)
print(df["id"].nunique())

In [ ]:
df = pd.DataFrame(data["results"])
print(df.shape)
df.head()

In [ ]:
print(len(all_jobs))

In [ ]:
df = pd.DataFrame(all_jobs)
print(df.shape)
print(df["id"].nunique())

In [ ]:
def fetch_jobs(keyword, location, max_pages=5):
    """取某个关键词在某地区的岗位，返回列表"""
    jobs = []
    for page in range(1, max_pages + 1):
        url = f"https://api.adzuna.com/v1/api/jobs/gb/search/{page}"
        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "results_per_page": 50,
            "what": keyword,
            "where": location,
        }
        r = requests.get(url, params=params)
        if r.status_code != 200:
            print(f"  page {page} failed: {r.status_code}")
            break
        
        results = r.json()["results"]
        if len(results) == 0:
            break
        
        for job in results:
            job["search_keyword"] = keyword
            job["search_location"] = location
        jobs.extend(results)
        
        time.sleep(1)
    return jobs

In [ ]:
test = fetch_jobs("data analyst", "london", max_pages=1)
print(len(test))
print(test[0]["search_keyword"], "|", test[0]["search_location"])

In [ ]:
test = fetch_jobs("data analyst", "london", max_pages=2)
print(len(test))

In [ ]:
keywords = [
    "data analyst",
    "data scientist",
    "machine learning engineer",
    "insight analyst",
    "graduate analyst",
]

all_jobs = []

for kw in keywords:
    print(f"fetching: {kw}")
    jobs = fetch_jobs(kw, "london", max_pages=5)
    print(f"  got {len(jobs)}")
    all_jobs.extend(jobs)

print(f"\n总计 {len(all_jobs)} 条")

In [ ]:
import os

os.makedirs("../data/raw", exist_ok=True)

df = pd.DataFrame(all_jobs)
df.to_csv("../data/raw/adzuna_london_20260806.csv", index=False)

print(df.shape)
print(df["id"].nunique())

In [ ]:
%pip install pandas

In [ ]:
keywords = [
    "data analyst",
    "data scientist",
    "machine learning engineer",
    "insight analyst",
    "graduate analyst",
]
locations = ["london", "manchester", "birmingham", "edinburgh", "bristol"]

def probe_count(keyword, location):
    """只问总数，不取数据"""
    url = "https://api.adzuna.com/v1/api/jobs/gb/search/1"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 1,
        "what": keyword,
        "where": location,
    }
    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"  {keyword} / {location} failed: {r.status_code}")
        return None
    return r.json()["count"]

counts = {}
for kw in keywords:
    for loc in locations:
        n = probe_count(kw, loc)
        counts[(kw, loc)] = n
        print(f"{kw:28s} {loc:12s} {n}")
        time.sleep(1)

In [ ]:
def probe_phrase(keyword, location):
    """用 what_phrase 精确短语匹配，只问总数"""
    url = "https://api.adzuna.com/v1/api/jobs/gb/search/1"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 1,
        "what_phrase": keyword,
        "where": location,
    }
    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"  {keyword} failed: {r.status_code}")
        return None
    return r.json()["count"]

print(f"{'keyword':28s} {'what':>8s} {'phrase':>8s}")
for kw in keywords:
    n_phrase = probe_phrase(kw, "london")
    n_what = counts[(kw, "london")]
    print(f"{kw:28s} {n_what:>8d} {n_phrase:>8d}")
    time.sleep(1)

In [ ]:
sample = fetch_jobs("data scientist", "london", max_pages=1)
print(len(sample))
print("-" * 60)
for job in sample:
    print(job["title"])

In [ ]:
url = "https://api.adzuna.com/v1/api/jobs/gb/search/40"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data scientist",
    "where": "london",
}
r = requests.get(url, params=params)
print(r.status_code)
deep = r.json()["results"]
print(len(deep))
print("-" * 60)
for job in deep:
    print(job["title"])

In [ ]:
all_jobs = []
for kw in keywords:
    for loc in locations:
        jobs = fetch_jobs(kw, loc, max_pages=5)
        print(f"{kw:28s} {loc:12s} {len(jobs):>4d}")
        all_jobs.extend(jobs)

print("-" * 50)
print("total:", len(all_jobs))

In [ ]:
n = probe_count("data analyst", "london")
print(n)

In [ ]:
import pandas as pd

df = pd.DataFrame(all_jobs)
print(df.shape)

df.to_csv("../data/raw/jobs_multi_location_20260808.csv", index=False)
print("saved")

In [ ]:
print(df["description"].str.len().describe())
print(df["description"].str.endswith("…").mean())
print(df["description"].iloc[0])